In [1]:
# pip -q install -U sentence-transformers spacy tqdm requests beautifulsoup4 readability-lxml faiss-cpu
# python -m spacy download en_core_web_sm

In [2]:
from pathlib import Path

ROOT = Path("..")
DATA_IN = ROOT / "dataset" / "unified_facts.json"
DATA_PRE = ROOT / "dataset" / "preprocessed.json"
CACHE = ROOT / "cache"
ARTIFACTS = ROOT / "artifacts"
CACHE.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(ROOT))

### Preprocess Data

In [3]:
from src.preprocess import preprocess

records = preprocess(in_path=DATA_IN, out_path=DATA_PRE, max_j_sents=5)
print("Preprocessed records:", len(records))

Preprocessed records: 27511


### Encoder Candidates and Classifier Wrapper

In [4]:
from src.embeddings import get_classifier_embeddings, get_rag_embeddings
clf_bases = ["dhruvpal/fake-news-bert", "bert-base-uncased", "roberta-base"]
rag_models = ["sentence-transformers/all-mpnet-base-v2", "facebook/dpr-question_encoder-single-nq-base", "sentence-transformers/all-MiniLM-L6-v2"]

for m in clf_bases:
    emb, ids = get_classifier_embeddings(m, preproc_json=DATA_PRE, out_dir=CACHE, bs=64, max_len=256)
    print("clf:", m, "->", emb.shape)

for r in rag_models:
    emb, ids = get_rag_embeddings(r, preproc_json=DATA_PRE, out_dir=CACHE, bs=64)
    print("rag:", r, "->", emb.shape)


clf: dhruvpal/fake-news-bert -> (27511, 768)
clf: bert-base-uncased -> (27511, 768)
clf: roberta-base -> (27511, 768)
rag: sentence-transformers/all-mpnet-base-v2 -> (27511, 768)
rag: facebook/dpr-question_encoder-single-nq-base -> (27511, 768)
rag: sentence-transformers/all-MiniLM-L6-v2 -> (27511, 384)


fine tuning

In [5]:
from src.ft_classify import fine_tune_base

ft_bert = fine_tune_base("bert-base-uncased", preproc_json=DATA_PRE, output_dir=None, epochs=3, per_device_train_batch_size=16, cache_dir=CACHE)
ft_roberta = fine_tune_base("roberta-base", preproc_json=DATA_PRE, output_dir=None, epochs=3, per_device_train_batch_size=16, cache_dir=CACHE)
print("Fine-tuned checkpoints:", ft_bert, ft_roberta)

Found existing checkpoint, skipping: ..\cache\models\ft_bert-base-uncased
Found existing checkpoint, skipping: ..\cache\models\ft_roberta-base
Fine-tuned checkpoints: ..\cache\models\ft_bert-base-uncased ..\cache\models\ft_roberta-base


re-extract embeddings from fine tuned model

In [6]:
from src.embeddings import get_classifier_embeddings

for ck in [ft_bert, ft_roberta]:
    print("Re-extracting embeddings for:", ck)
    embs, ids = get_classifier_embeddings(ck, preproc_json=DATA_PRE, out_dir=CACHE, bs=64, max_len=256)
    print(" -> saved embeddings shape:", embs.shape, "ids:", len(ids))

Re-extracting embeddings for: ..\cache\models\ft_bert-base-uncased
 -> saved embeddings shape: (27511, 768) ids: 27511
Re-extracting embeddings for: ..\cache\models\ft_roberta-base
 -> saved embeddings shape: (27511, 768) ids: 27511


### Evaluate embeddings

In [7]:
import csv
from src.embeddings import get_classifier_embeddings, get_rag_embeddings
from src.eval_utils import eval_embedding_clf, rag_self_retrieval_eval

clf_candidates = ["dhruvpal/fake-news-bert", ft_bert, "bert-base-uncased", ft_roberta, "roberta-base"]
rag_candidates = ["sentence-transformers/all-mpnet-base-v2", "facebook/dpr-question_encoder-single-nq-base", "sentence-transformers/all-MiniLM-L6-v2"]

rows = []
for m in clf_candidates:
    embs, ids = get_classifier_embeddings(m, preproc_json=DATA_PRE, out_dir=CACHE, bs=64, max_len=256)
    stats, _ = eval_embedding_clf(embs, ids, preproc_json=DATA_PRE)
    rows.append({"model": m, "role": "clf", **stats, "dim": embs.shape[1]})
    print("clf eval:", m, stats)

for r in rag_candidates:
    embs, ids = get_rag_embeddings(r, preproc_json=DATA_PRE, out_dir=CACHE, bs=64)
    stats = rag_self_retrieval_eval(embs, ids, topk=5)
    rows.append({"model": r, "role": "rag", **stats, "dim": embs.shape[1]})
    print("rag eval:", r, stats)


all_keys = set()
for r in rows:
    all_keys.update(r.keys())
fieldnames = sorted(all_keys)

csv_path = ARTIFACTS / "embedding_eval_summary.csv"
with open(csv_path, "w", newline="", encoding="utf8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow({k: r.get(k, "") for k in fieldnames})

print("Saved evaluation summary to", csv_path)

clf eval: dhruvpal/fake-news-bert {'acc': 0.6830819552971107, 'f1': 0.47501505117399156, 'roc_auc': 0.6906326121574484, 'pr_auc': 0.5833920441043111}
clf eval: ..\cache\models\ft_bert-base-uncased {'acc': 0.8526258404506633, 'f1': 0.7853929611008204, 'roc_auc': 0.8993624652846832, 'pr_auc': 0.8718125002357207}
clf eval: bert-base-uncased {'acc': 0.7148827912047974, 'f1': 0.5510729613733906, 'roc_auc': 0.7254652804518034, 'pr_auc': 0.6431943223712387}
clf eval: ..\cache\models\ft_roberta-base {'acc': 0.8233690714155915, 'f1': 0.7385691231845078, 'roc_auc': 0.8768978294199705, 'pr_auc': 0.8501550113601953}
clf eval: roberta-base {'acc': 0.7096129383972378, 'f1': 0.5286135693215339, 'roc_auc': 0.7236195328224592, 'pr_auc': 0.6357764574140969}
rag eval: sentence-transformers/all-mpnet-base-v2 {'MRR': 0.999000399840064, 'Recall@1': 0.9983279415506525, 'Recall@5': 1.0}
rag eval: facebook/dpr-question_encoder-single-nq-base {'MRR': 0.9990428071195763, 'Recall@1': 0.9984006397441023, 'Recall@5

### FAISS Index and query

In [8]:
import faiss, numpy as np, torch
from src.embeddings import get_rag_embeddings, build_faiss_index, is_dpr_model
from src.preprocess import preprocess

rag_model = rag_models[0]
embs, doc_ids = get_rag_embeddings(rag_model, preproc_json=DATA_PRE, out_dir=CACHE, bs=64)
index = build_faiss_index(embs)

records = preprocess(in_path=DATA_IN, out_path=DATA_PRE)
queries = [records[i]["statement"] for i in range(min(5, len(records)))]

# encode queries
if is_dpr_model(rag_model):
    from transformers import AutoTokenizer, DPRQuestionEncoder
    tok = AutoTokenizer.from_pretrained(rag_model, local_files_only=True) if Path(rag_model).exists() else AutoTokenizer.from_pretrained(rag_model)
    model = DPRQuestionEncoder.from_pretrained(rag_model, local_files_only=True) if Path(rag_model).exists() else DPRQuestionEncoder.from_pretrained(rag_model)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    q_embs = []
    for i in range(0, len(queries), 16):
        batch = queries[i:i+16]
        enc = tok(batch, truncation=True, padding=True, return_tensors="pt", max_length=256)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, return_dict=True)
            q_embs.append(out.pooler_output.cpu().numpy())
    q_embs = np.vstack(q_embs).astype("float32")
else:
    from sentence_transformers import SentenceTransformer
    rag_encoder = SentenceTransformer(rag_model, device=("cuda" if torch.cuda.is_available() else "cpu"))
    q_embs = rag_encoder.encode(queries, convert_to_numpy=True).astype("float32")

faiss.normalize_L2(q_embs)
D, I = index.search(q_embs, 5)
for qi, inds in enumerate(I):
    print("Query:", queries[qi])
    print("Top hits:", [doc_ids[i] for i in inds])
    print()

Query: 90 percent of Americans "support universal background checks" for gun purchases.
Top hits: ['liar2_13847', 'liar2_14333', 'liar2_11109', 'liar2_10798', 'liar2_7068']

Query: Last year was one of the deadliest years ever for law enforcement officers.
Top hits: ['liar2_13411', 'liar2_18308', 'liar2_10741', 'liar2_18497', 'liar2_12386']

Query: Bernie Sanders's plan is "to raise your taxes to 90 percent.
Top hits: ['liar2_10882', 'liar2_10851', 'liar2_17124', 'snopes_2269', 'liar2_18324']

Query: Voter ID is supported by an overwhelming majority of NYers, from all across the state, walks of life, & political parties.
Top hits: ['liar2_20697', 'liar2_14747', 'liar2_7021', 'liar2_1362', 'liar2_18785']

Query: Says Barack Obama "robbed Medicare (of) $716 billion to pay for ... Obamacare.
Top hits: ['liar2_6095', 'liar2_10582', 'liar2_4243', 'liar2_3766', 'liar2_6184']

